In [4]:
import torch
from torch import nn
from torchvision.transforms import v2
#import torchaudio
from torchvision import tv_tensors

import os
import glob

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

import numpy as np
import pandas as pd

In [ ]:
class DummyMultimodalNN(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return x
    
class MultimodalDataset(torch.utils.data.Dataset):
    label_dict = {'NOAGGR':0, 'AGGR':1}
    def __init__(
            self,
            time_intervals_df: pd.DataFrame,
            path_to_dataset: str,
            modality_augmentation_dict: dict,
            actual_modalities_list: list,
            device: torch.device,
            text_embedding_type: str,
            modality2aggr = {'video':'phys', 'text':'verb', 'audio':'verb'},
            video_shape=(1, 3, 112, 112),
            audio_shape=(1,),
            text_shape=(1, 768)
            ):
        '''
        time_intervals_df - pandas dataframe с информацией о датасете
            (поля: project_name;unique_person;video;person_name;gender;phys_t1;phys_t2;text;aggr_type;phys_aggr_label;video_id;person_id;cluster_id;verb_t1;verb_t2;verb_aggr_label)
        path_to_dataset - путь до корневого каталога с данными
        modality_augmentation_dict - словарь со структурой {'имя модальности': torchvision.transforms}
        actual_modalities_list - список с именами используемых в обработке модальностей
        device - вычислительное устройство
        text_embedding_type - срока с именем типа векторных представлений слов (BERT, RoBERT и т.д.) для определения верного пути до них
        modality2aggr = {'video':'phys', 'text':'verb', 'audio':'verb'} - словарь, содержащий отображение модальности на тип агрессии
        video_shape=(1, 3, 112, 112) - размер кадра, обрабатываемого нейросетью
        audio_shape=(1,) - размер аудиосиганала, обрабатываемого нейросетью 
        text_shape=(1, 768) - размер векторных представлений слов, обрабатываемых нейронной сетью

        '''
        super().__init__()
        self.modality_augmentation_dict = modality_augmentation_dict
        self.modality2aggr = modality2aggr
        self.path_to_dataset = path_to_dataset
        self.time_intervals_df = time_intervals_df
        self.actual_modalities_list = actual_modalities_list
        self.device = device
        self.text_embedding_type = text_embedding_type
        self.video_shape = video_shape
        self.audio_shape = audio_shape
        self.text_shape = text_shape
    
    def __len__(self):
        return len(self.time_intervals_df)
        
    def __getitem__(self, idx):
        #!!!!
        #data_entry = self.time_intervals_df.iloc[idx]
        data_entry = self.time_intervals_df.loc[idx]
        aggr_type = data_entry['aggr_type']
        cluster_id = data_entry['cluster_id']
        video_id = data_entry['video_id']
        phys_t1 = data_entry['phys_t1']
        phys_t2 = data_entry['phys_t2']
        verb_t1 = data_entry['verb_t1']
        verb_t2 = data_entry['verb_t2']
        person_id = data_entry['person_id']
        phys_label = data_entry['phys_aggr_label']
        verb_label = data_entry['verb_aggr_label']

        multimodal_data_dict = {}
        multimodal_label_dict = {}
        # заполняем пустыми значениями, чтобы правильно работал DataLoader
        for modality in self.actual_modalities_list:
            if modality =='text':
                text = torch.full(self.text_shape, fill_value=-1., device=self.device)
                text = self.modality_augmentation_dict['text'](text)
                multimodal_data_dict['text'] = text
                multimodal_label_dict['text'] = torch.as_tensor(-1, dtype=torch.int64, device=self.device)
            elif modality == 'audio':
                audio = torch.full(self.audio_shape, fill_value=-1., device=self.device)
                audio = self.modality_augmentation_dict['audio'](audio)
                multimodal_data_dict['audio'] = audio
                multimodal_label_dict['audio'] = torch.as_tensor(-1, dtype=torch.int64, device=self.device)
            elif modality == 'video':
                video = torch.full(self.video_shape, fill_value=-1., device=self.device)
                video = tv_tensors.Video(video, device=self.device)
                video = self.modality_augmentation_dict['video'](video)
                multimodal_data_dict['video'] = video.permute((1, 0, 2, 3))
                multimodal_label_dict['video'] = torch.as_tensor(-1, dtype=torch.int64, device=self.device)
        is_video = False
        is_audio = False
        is_text = False
        for modality in self.actual_modalities_list:
            if aggr_type == 'verb':
                verb_name = f'c-{cluster_id}_{video_id}_{person_id}_{verb_t1/1000}-{verb_t2/1000}_{verb_label}'
                if modality == 'text':
                    path_to_text = os.path.join(self.path_to_dataset, 'verbal', self.text_embedding_type, f'{verb_name}.npy')
                    text = torch.as_tensor(np.load(path_to_text), dtype=torch.float32, device=self.device)
                    text = self.modality_augmentation_dict['text'](text)
                    multimodal_data_dict['text'] = text
                    multimodal_label_dict['text'] = torch.as_tensor(self.label_dict[verb_label], dtype=torch.int64, device=self.device)
                    is_text = True
                elif modality == 'audio':
                    path_to_audio = os.path.join(self.path_to_dataset, 'verbal', 'pt_waveform', f'{verb_name}.pt')
                    audio = torch.load(path_to_audio).to(self.device)
                    audio = self.modality_augmentation_dict['audio'](audio)
                    multimodal_data_dict['audio'] = audio
                    multimodal_label_dict['audio'] = torch.as_tensor(self.label_dict[verb_label], dtype=torch.int64, device=self.device)
                    is_audio = True
            elif aggr_type == 'phys':
                if modality == 'video':
                    phys_name = f'c-{cluster_id}_{video_id}_{person_id}_{phys_t1/1000}-{phys_t2/1000}_{phys_label}'
                    path_to_video = os.path.join(self.path_to_dataset, 'physical', 'video', f'{phys_name}.pt')
                    video = torch.load(path_to_video)
                    video = tv_tensors.Video(video, device=self.device)
                    video = self.modality_augmentation_dict['video'](video)
                    multimodal_data_dict['video'] = video.permute((1, 0, 2, 3))
                    multimodal_label_dict['video'] = torch.as_tensor(self.label_dict[phys_label], dtype=torch.int64, device=self.device)
                    is_video = True
            elif aggr_type == 'phys&verb':
                verb_name = f'c-{cluster_id}_{video_id}_{person_id}_{verb_t1/1000}-{verb_t2/1000}_{verb_label}'
                phys_name = f'c-{cluster_id}_{video_id}_{person_id}_{phys_t1/1000}-{phys_t2/1000}_{phys_label}'
                if modality == 'text':
                    path_to_text = os.path.join(self.path_to_dataset, 'verbal', self.text_embedding_type, f'{verb_name}.npy')
                    text = torch.as_tensor(np.load(path_to_text), dtype=torch.float32, device=self.device)
                    text = self.modality_augmentation_dict['text'](text)
                    multimodal_data_dict['text'] = text
                    multimodal_label_dict['text'] = torch.as_tensor(self.label_dict[verb_label], dtype=torch.int64, device=self.device)
                    is_text = True
                elif modality == 'audio':
                    path_to_audio = os.path.join(self.path_to_dataset, 'verbal', 'pt_waveform', f'{verb_name}.pt')
                    audio = torch.load(path_to_audio).to(self.device)
                    audio = self.modality_augmentation_dict['audio'](audio)
                    multimodal_data_dict['audio'] = audio
                    multimodal_label_dict['audio'] = torch.as_tensor(self.label_dict[verb_label], dtype=torch.int64, device=self.device)
                    is_audio = True
                elif modality == 'video':
                    path_to_video = os.path.join(self.path_to_dataset, 'physical', 'video', f'{phys_name}.pt')
                    video = torch.load(path_to_video)
                    video = tv_tensors.Video(video, device=self.device)
                    video = self.modality_augmentation_dict['video'](video)
                    multimodal_data_dict['video'] = video.permute((1, 0, 2, 3))
                    multimodal_label_dict['video'] = torch.as_tensor(self.label_dict[phys_label], dtype=torch.int64, device=self.device)
                    is_video = True

        output_data_list = []
        for modality, tensor in multimodal_data_dict.items():
            if modality == 'audio':
                if not is_audio:
                    modality = 'audio_EMPTY'
            elif modality == 'video':
                if not is_video:
                    modality = 'video_EMPTY'
            elif modality == 'text':
                if not is_text:
                    modality = 'text_EMPTY'
            output_data_list.append((modality, tensor))

        output_labels_list = []
        for modality, label in multimodal_label_dict.items():
            if modality == 'audio':
                if not is_audio:
                    modality = 'audio_EMPTY'
            elif modality == 'video':
                if not is_video:
                    modality = 'video_EMPTY'
            elif modality == 'text':
                if not is_text:
                    modality = 'text_EMPTY'
            output_labels_list.append((modality, label))
            
        return tuple(output_data_list), tuple(output_labels_list)
